# T3-05 · Chọn silver pool và chuẩn bị bundles
Chạy trên Colab CPU. Notebook này thực hiện bước 1–3: chọn ID, kiểm tra dữ liệu MS1/MS2 và lưu bundle. Chưa gán nhãn. Kết quả nằm trong thư mục Drive riêng; chạy lại sẽ dùng lại bundle có hash khớp.


In [ ]:
# @title 1. Kết nối Drive và lấy code từ nhánh T3-05
from google.colab import drive
drive.mount('/content/drive')
import subprocess, sys
from pathlib import Path
REPO = Path('/content/arxiv-data-science')
BRANCH = 'feature/m3-tv3-auto-labeling'
if not REPO.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', 'https://github.com/LeoTkiet/arxiv-data-science.git', str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', BRANCH], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO) + '[matching,reporting]'], check=True)
print('Code:', subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip())


In [ ]:
# @title 2. Chọn 2.000 bài và danh sách dự phòng
from arxiv_pipeline.matching.silver_pool import select_pool
POPULATION = Path('/content/drive/MyDrive/T3-05/population_manifest.json') # @param {type:'string'}
OUTPUT = Path('/content/drive/MyDrive/T3-05/output') # @param {type:'string'}
TARGET_COUNT = 2000 # @param {type:'integer'}
SEED = 42 # @param {type:'integer'}
selection = select_pool(Path(POPULATION), REPO/'configs/ms1_group6_ranges.json', Path(OUTPUT)/'selection.json', seed=SEED, target_count=TARGET_COUNT)
print('Đã chọn:', sum(map(len, selection['selected'].values())), '| quota:', selection['minimum_auto_count'])
print('Dự phòng:', sum(map(len, selection['reserves'].values())))


In [ ]:
# @title 3. Kiểm tra MS1/MS2 và tạo bundles (có thể chạy lại)
import json
from arxiv_pipeline.matching.silver_pool import prepare_pool
# Tạo roots.json trên Drive: {"tv1": {"ms1_root": ".../tv1/data", "ms2_root": ".../tv1"}, ...}
ROOTS = Path('/content/drive/MyDrive/T3-05/roots.json') # @param {type:'string'}
result = prepare_pool(Path(OUTPUT)/'selection.json', Path(ROOTS), Path(OUTPUT)/'prepared')
print('Ready:', result['ready'], '| Chưa ready:', result['not_ready'])
print('Manifest:', Path(OUTPUT)/'prepared/prepared_manifest.json')
